# Encrypted RAG Chatbot with CyborgDB

## 🚀 Quick Start Guide

This notebook is designed to run on **Google Colab** and will automatically set up everything you need!

---

### Step 1: Choose Your Database

In the first code cell below, set your database type:
- `CYBORGDB_DB_TYPE = 'postgres'` for PostgreSQL
- `CYBORGDB_DB_TYPE = 'redis'` for Redis

The notebook will automatically install and configure the selected database.

---

### Step 2: Add API Keys

You'll an **OpenAI API key** (get it from https://platform.openai.com/api-keys) to use a hosted LLM for demo purposes.

You can set it as the `OPENAI_API_KEY` environment variable, or be prompted when you run the notebook.

---

### Step 3: Run All Cells

Simply click **Runtime → Run all** and the notebook will:
- ✅ Install all Python dependencies
- ✅ After install, **manually click "Runtime → Restart and run all"** to continue
- ✅ Install your chosen database (PostgreSQL or Redis)
- ✅ Configure the database connection
- ✅ Start the CyborgDB service
- ✅ Launch the Gradio chatbot interface

**Note:** The first run will automatically restart the runtime after installing packages. This prevents version conflicts. Just run all cells again after the restart!

---

### Step 4: Open the Public URL

When the last cell finishes running, you'll see a **public Gradio URL** (e.g., `https://xxxxx.gradio.live`).

**Click the URL** to open the chatbot interface in a new tab, then:
1. Upload PDF or TXT documents
2. Click "Create Encrypted Vector Store"
3. Start asking questions about your documents!

All vector embeddings are encrypted end-to-end.

In [ ]:
# 0. Install dependencies and handle automatic restart if needed

import subprocess, sys, os

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def restart():
    if IN_COLAB:
        print("🔄 Restarting Colab runtime...")
        print("Please click on \"Runtime > Restart and run all\" if it does not restart automatically.")
        
        # Wait for a moment to let the print flush
        import time
        time.sleep(2)

    else:
        from IPython import get_ipython
        print("🔄 Restarting Jupyter kernel...")
        print("Please re-run all notebook cells if it does not restart automatically.")
        get_ipython().kernel.do_shutdown(restart=True)
    
    raise SystemExit(0)

reqs = [
    "numpy","cyborgdb[langchain]>=0.13.0","getpass4",
    "cyborgdb-service","scipy>=1.14","scikit-learn>=1.5",
    "transformers>=4.41","sentence-transformers>=3.1",
    "langchain==0.3.26","langchain-community==0.3.27",
    "langchain-huggingface==0.3.1","langchain-openai==0.3.28",
    "pypdf==5.8.0","openai==1.97.1","gradio==5.38.0",
    "cryptography"
]

# Check if any dependencies need to be installed or updated
print("📦 Checking dependencies...")
probe = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--dry-run", *reqs],
    capture_output=True, text=True
)
out = probe.stdout + probe.stderr

# If pip indicates it would install or update anything, do so and restart
if "Would install" in out or "Installing collected" in out:
    print("⬇️ Dependencies missing or outdated — installing and restarting...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", *reqs], check=True)
    restart()
else:
    print("✅ All requirements already satisfied — no restart needed.")


In [ ]:
# 1. Configure CyborgDB database type

# Choose your database type: 'postgres' or 'redis'
CYBORGDB_DB_TYPE = 'redis'

In [ ]:
# 2. Install and Setup PostgreSQL or Redis

import subprocess
import platform
import time
import getpass

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, check=False)

# Detect OS
is_mac = platform.system() == "Darwin"
is_linux = platform.system() == "Linux"

if CYBORGDB_DB_TYPE == 'redis':
    # Install Redis if needed
    if not run("which redis-server").stdout:
        print("⬇️ Redis not found — installing...")
        if is_mac:
            run("brew install redis >/dev/null 2>&1")
        elif is_linux:
            run("sudo apt-get update >/dev/null 2>&1")
            run("sudo apt-get install -y redis-server >/dev/null 2>&1")

    # Always attempt to start the service
    if is_mac:
        run("brew services start redis >/dev/null 2>&1")
    elif is_linux:
        run("sudo service redis-server start")

    # Wait a bit for Redis to start
    time.sleep(3)

    # Setup connection parameters
    REDIS_HOST = "localhost"
    REDIS_PORT = 6379
    REDIS_DB = 0

    CYBORGDB_CONNECTION_STRING = f"host:{REDIS_HOST},port:{REDIS_PORT},db:{REDIS_DB}"

    print("✅ Redis ready on localhost:6379")

elif CYBORGDB_DB_TYPE == 'postgres':
    # Install PostgreSQL if needed
    if not run("which psql").stdout:
        print("⬇️ PostgreSQL not found — installing...")
        if is_mac:
            run("brew install postgresql@14 >/dev/null 2>&1")
        elif is_linux:
            run("sudo apt-get update >/dev/null 2>&1")
            run("sudo apt-get install -y postgresql postgresql-contrib >/dev/null 2>&1")

    # Always attempt to start the service
    if is_mac:
        run("brew services start postgresql@14 >/dev/null 2>&1")
    elif is_linux:
        run("sudo service postgresql start")

    # Wait a bit for PostgreSQL to start
    time.sleep(3)

    # Setup connection parameters
    POSTGRES_HOST = "localhost"
    POSTGRES_PORT = 5432
    POSTGRES_DB = "postgres"
    POSTGRES_USER = getpass.getuser()
    POSTGRES_PASSWORD = "password"
    CYBORGDB_CONNECTION_STRING = f"host={POSTGRES_HOST} port={POSTGRES_PORT} dbname={POSTGRES_DB} user={POSTGRES_USER} password={POSTGRES_PASSWORD}"

    # Create user (if needed)
    if is_linux:
        run(f"sudo -u postgres psql -c \"CREATE USER {POSTGRES_USER} WITH PASSWORD '{POSTGRES_PASSWORD}' SUPERUSER;\" 2>/dev/null")
    else:
        run(f"psql postgres -c \"CREATE USER {POSTGRES_USER} WITH PASSWORD '{POSTGRES_PASSWORD}';\" 2>/dev/null")
        run(f"psql postgres -c \"ALTER USER {POSTGRES_USER} WITH SUPERUSER;\" 2>/dev/null")

    print(f"✅ PostgreSQL ready: {POSTGRES_USER}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}")

else:
    raise ValueError("CYBORGDB_DB_TYPE must be either 'postgres' or 'redis'")

In [ ]:
# 3. Configure API keys and environment variables

import os
import getpass
from cyborgdb import get_demo_api_key

# Configure API key or demo API key (expires in 1 hour)
CYBORGDB_API_KEY = os.environ.get("CYBORGDB_API_KEY") or get_demo_api_key()
os.environ["CYBORGDB_API_KEY"] = CYBORGDB_API_KEY

# Get OpenAI API key from environment variable or prompt
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or getpass.getpass("Enter your OpenAI API Key: ")

# Set environment variables
os.environ["CYBORGDB_API_KEY"] = CYBORGDB_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["CYBORGDB_DB_TYPE"] = CYBORGDB_DB_TYPE
os.environ["CYBORGDB_CONNECTION_STRING"] = CYBORGDB_CONNECTION_STRING
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "false"

In [ ]:
# 4. Start CyborgDB service and wait for it to be ready

import subprocess
import time
import requests

# Start service
print("🔄 Starting CyborgDB service...")
log_file = "cyborgdb_service.log"

process = subprocess.Popen(
    "cyborgdb-service",
    stdout=open(log_file, "w"),
    stderr=subprocess.STDOUT,
    env=dict(os.environ)
)

# Wait for startup
for i in range(60):
    try:
        response = requests.get("http://localhost:8000/v1/health", timeout=1)
        if response.status_code == 200:
            print("\n✅ CyborgDB service started!")
            break
    except:
        pass
    time.sleep(1)
    if i % 5 == 0 and i > 0:
        print(f"   Waiting... ({i}/60)")
else:
    print("\n❌ Service failed to start. Logs:")
    !tail -30 {log_file}
    raise RuntimeError("Failed to start service")

In [ ]:
# 5. Main Application Code

import os
import pathlib
import base64
import uuid
import numpy as np
import langchain.chains
import langchain.chains.combine_documents
import langchain.docstore.document
import langchain.document_loaders
import langchain.prompts
import langchain.text_splitter
import langchain_community.chat_message_histories
import langchain_core.runnables.history
import langchain_huggingface
import langchain_openai
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import cyborgdb
from cyborgdb.integrations.langchain import CyborgVectorStore
import gradio as gr

print("✅ Libraries imported")

# Settings
EMBEDDING_MODEL_ID = "all-MiniLM-L6-v2"
OPENAI_MODEL = "gpt-4o-mini"
OPENAI_URL = None
CHUNK_SIZE = 2048
CHUNK_OVERLAP = 128
CYBORGDB_HOST = "http://localhost:8000"
CYBORGDB_KEYS = {}
VECTORSTORE_STORE = {}
CHAT_HISTORY_STORE = {}
CHAINS_STORE = {}
LAST_QUERY_INFO = {}
MOST_RECENT_PROMPT = None

print("✅ Settings configured")

# Functions
def split_documents(file_path: str) -> list:
    file = pathlib.Path(file_path)
    if not file.exists():
        raise ValueError("File not found")
    if file.suffix == ".pdf":
        loader = langchain.document_loaders.PyPDFLoader(str(file))
        pages = loader.load()
        splitter = langchain.text_splitter.RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
        return splitter.split_documents(pages)
    else:
        loader = langchain.document_loaders.TextLoader(str(file))
        pages = loader.load()
        splitter = langchain.text_splitter.RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
        return splitter.split_documents(pages)

def generate_encryption_key() -> bytes:
    """Generate a 256-bit AES-GCM key."""
    return AESGCM.generate_key(bit_length=256)

def encrypt_chunk(text: str, key: bytes) -> str:
    aesgcm = AESGCM(key)
    nonce = os.urandom(12)
    ciphertext = aesgcm.encrypt(nonce, text.encode(), None)
    return base64.b64encode(nonce + ciphertext).decode()

def load_vectorstore(session_id: str) -> CyborgVectorStore:
    if session_id in VECTORSTORE_STORE:
        return VECTORSTORE_STORE[session_id]
    if session_id not in CYBORGDB_KEYS:
        # Generate key using cryptography library directly
        CYBORGDB_KEYS[session_id] = generate_encryption_key()
    vector_store = CyborgVectorStore(
        index_name=f"session_{session_id}",
        index_key=CYBORGDB_KEYS[session_id],
        api_key=CYBORGDB_API_KEY,
        base_url=CYBORGDB_HOST,
        embedding=EMBEDDING_MODEL_ID,
        index_type="ivfflat",
        metric="cosine",
    )
    VECTORSTORE_STORE[session_id] = vector_store
    return vector_store

def upload_documents(file_paths: list[str], session_id: str) -> str:
    try:
        if not file_paths:
            return "❌ No files selected"

        print(f"Processing {len(file_paths)} files for session {session_id}")
        vectorstore = load_vectorstore(session_id)
        encryption_key = bytes(CYBORGDB_KEYS[session_id])
        all_docs = []
        previews = []

        for file_path in file_paths:
            print(f"Processing: {file_path}")
            docs = split_documents(file_path)
            all_docs.extend(docs)
            for i, doc in enumerate(docs[:3]):
                encrypted = encrypt_chunk(doc.page_content, encryption_key)
                previews.append(f"  Chunk {i+1}: {encrypted[:60]}...")

        if all_docs:
            print(f"Adding {len(all_docs)} documents to vector store")
            vectorstore.add_documents(all_docs)
            print("Training index...")
            vectorstore.index.train()
            print("Index trained successfully")

        result = [
            f"✅ Processed {len(file_paths)} file(s) with {len(all_docs)} chunks",
            "",
            "🔐 Encrypted chunk previews:"
        ]
        result.extend(previews[:5])
        if len(all_docs) > 5:
            result.append(f"  ... and {len(all_docs) - 5} more encrypted chunks")
        return "\n".join(result)
    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        print(f"ERROR in upload_documents: {error_msg}")
        return f"❌ Error: {str(e)}\n\nSee logs for details."

def load_chain(session_id: str, system_prompt: str):
    print(f"Loading chain for session {session_id}")
    llm = langchain_openai.ChatOpenAI(
        base_url=OPENAI_URL,
        model=OPENAI_MODEL,
        max_tokens=500,
        temperature=0.7
    )
    vectordb = load_vectorstore(session_id)
    retriever = vectordb.as_retriever(search_kwargs={"k": 3})

    contextualize_prompt = langchain.prompts.ChatPromptTemplate.from_messages([
        ("system", "Given chat history and a new question, reformulate the question to be standalone. If it's already standalone, return as is."),
        langchain.prompts.MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])

    history_aware_retriever = langchain.chains.create_history_aware_retriever(
        llm, retriever, contextualize_prompt
    )

    qa_prompt = langchain.prompts.ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        langchain.prompts.MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])

    document_chain = langchain.chains.combine_documents.create_stuff_documents_chain(llm, qa_prompt)
    rag_chain = langchain.chains.create_retrieval_chain(history_aware_retriever, document_chain)

    return langchain_core.runnables.history.RunnableWithMessageHistory(
        rag_chain,
        get_session_history=lambda: CHAT_HISTORY_STORE.get(
            session_id,
            langchain_community.chat_message_histories.ChatMessageHistory()
        ),
        input_messages_key="input",
        history_messages_key="chat_history",
        output_messages_key="answer",
    )

def create_comparison_html(query: str, result_docs=None, plaintext_embeddings=None, encrypted_embeddings=None):
    """Create HTML showing plaintext vs encrypted vectors."""
    if not result_docs or not plaintext_embeddings:
        return f"""
        <div style="margin: 20px 0; border: 2px solid #e2e8f0; border-radius: 12px; overflow: hidden; background: white; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
            <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; text-align: center;">
                <h2 style="color: white; margin: 0; font-size: 24px;">🔒 Retrieved Results: Plaintext vs Encrypted Vectors</h2>
            </div>
            <div style="padding: 40px; text-align: center; color: #666;">
                <p style="font-size: 16px;">Query: "<strong>{query}</strong>"</p>
                <p style="margin-top: 16px; color: #e53e3e; font-weight: 600;">⚠️ No documents found. Please upload documents first!</p>
            </div>
        </div>
        """

    results_html = ""
    for i, (doc, plaintext_emb, encrypted_emb) in enumerate(zip(result_docs, plaintext_embeddings, encrypted_embeddings)):
        content_preview = doc[:150] if len(doc) > 150 else doc
        plaintext_preview = "[" + ", ".join([f"{x:.4f}" for x in plaintext_emb[:20]]) + ", ...]"
        encrypted_preview = encrypted_emb[:150] + "..." if len(encrypted_emb) > 150 else encrypted_emb

        results_html += f"""
        <div style="margin-bottom: 16px; padding: 14px; background: #fafafa; border-radius: 8px; border: 1px solid #e2e8f0;">
            <div style="font-weight: 600; color: #333; margin-bottom: 10px; font-size: 15px;">
                📄 Result #{i+1}
            </div>
            <div style="margin-bottom: 12px; padding: 10px; background: white; border-radius: 6px; border: 1px solid #e2e8f0;">
                <div style="font-size: 14px; color: #666; font-weight: 600; margin-bottom: 4px;">
                    📝 Document Content:
                </div>
                <div style="font-size: 14px; color: #2d3748; line-height: 1.5;">
                    "{content_preview}..."
                </div>
            </div>
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px;">
                <div style="padding: 10px; background: #fff5f5; border: 2px solid #fc8181; border-radius: 6px;">
                    <div style="font-size: 14px; color: #c53030; font-weight: 600; margin-bottom: 6px;">
                        🔓 Plaintext Vector (first 20 dims):
                    </div>
                    <div style="background: white; padding: 6px; border-radius: 4px; font-family: 'Courier New', monospace; font-size: 14px; color: #4a5568; max-height: 120px; overflow-y: auto; line-height: 1.6; word-break: break-all;">
                        {plaintext_preview}
                    </div>
                </div>
                <div style="padding: 10px; background: #f0fff4; border: 2px solid #9ae6b4; border-radius: 6px;">
                    <div style="font-size: 14px; color: #22543d; font-weight: 600; margin-bottom: 6px;">
                        🔒 Encrypted Vector (first 150 chars):
                    </div>
                    <div style="background: white; padding: 6px; border-radius: 4px; font-family: 'Courier New', monospace; font-size: 14px; color: #4a5568; max-height: 120px; overflow-y: auto; word-break: break-all; line-height: 1.6;">
                        {encrypted_preview}
                    </div>
                </div>
            </div>
        </div>
        """

    return f"""
    <div style="--body-text-color: #1a202c; margin: 20px 0; border: 2px solid #e2e8f0; border-radius: 12px; overflow: hidden; background: white; 
  box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; text-align: center;">
            <h2 style="color: white; margin: 0; font-size: 24px;">🔒 Retrieved Results: Plaintext vs Encrypted Vectors</h2>
        </div>
        <div style="padding: 16px; background: #f9fafb;">
            <div style="font-size: 15px; color: #666; margin-bottom: 16px; padding: 8px; background: white; border-radius: 6px;">
                <strong>Your Query:</strong> "{query}"
            </div>
            <div style="margin-bottom: 16px; padding: 16px; background: #edf2f7; border-radius: 8px;">
                <div style="text-align: center; margin-bottom: 12px;">
                    <p style="margin: 0; color: #2d3748; font-size: 16px; font-weight: 600;">
                        💡 Why Encrypt Embeddings?
                    </p>
                </div>
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 16px;">
                    <div style="background: white; padding: 12px; border-radius: 6px; border-left: 4px solid #fc8181;">
                        <div style="font-size: 14px; color: #c53030; font-weight: 600; margin-bottom: 8px;">
                            🚨 The Problem
                        </div>
                        <ul style="margin: 0; padding-left: 20px; font-size: 14px; color: #1a202c; line-height: 1.8;">
                            <li>Embeddings capture semantic meaning of your documents</li>
                            <li>Machine learning attacks can reconstruct original text</li>
                            <li>Database breaches expose sensitive information</li>
                            <li>Compliance regulations require data protection</li>
                        </ul>
                    </div>
                    <div style="background: white; padding: 12px; border-radius: 6px; border-left: 4px solid #9ae6b4;">
                        <div style="font-size: 14px; color: #22543d; font-weight: 600; margin-bottom: 8px;">
                            ✅ The Solution
                        </div>
                        <ul style="margin: 0; padding-left: 20px; font-size: 14px; color: #1a202c; line-height: 1.8;">
                            <li>CyborgDB encrypts vectors before storing them</li>
                            <li>Homomorphic encryption enables search on encrypted data</li>
                            <li>Breaches only expose encrypted, meaningless data</li>
                            <li>Maintains full RAG functionality with zero trust</li>
                        </ul>
                    </div>
                </div>
            </div>
            {results_html}
        </div>
    </div>
    """

def chat(message: str, history, session: str, system_prompt: str) -> tuple[str, str]:
    global MOST_RECENT_PROMPT, LAST_QUERY_INFO
    MOST_RECENT_PROMPT = message

    try:
        print(f"\n=== Chat Query ===")
        print(f"Session: {session}")
        print(f"Message: {message}")

        if session not in CHAINS_STORE:
            if session not in CHAT_HISTORY_STORE:
                CHAT_HISTORY_STORE[session] = langchain_community.chat_message_histories.ChatMessageHistory()
            CHAINS_STORE[session] = load_chain(session, system_prompt)

        chain = CHAINS_STORE[session]
        vectorstore = load_vectorstore(session)
        encryption_key = bytes(CYBORGDB_KEYS[session])

        # Retrieve documents
        print("Retrieving documents...")
        retrieved_docs = vectorstore.similarity_search(message, k=3)
        print(f"Retrieved {len(retrieved_docs)} documents")

        # Process retrieved documents
        result_contents = []
        plaintext_embeddings = []
        encrypted_embeddings = []

        for i, doc in enumerate(retrieved_docs):
            content = doc.page_content
            result_contents.append(content)

            # Get plaintext embedding
            doc_plaintext_embedding = vectorstore.get_embeddings(content)
            plaintext_embeddings.append(doc_plaintext_embedding.tolist())

            # Create encrypted version
            doc_encrypted_embedding = encrypt_chunk(str(doc_plaintext_embedding.tolist()), encryption_key)
            encrypted_embeddings.append(doc_encrypted_embedding)

        # Store for display
        LAST_QUERY_INFO = {
            'query': message,
            'result_contents': result_contents,
            'plaintext_embeddings': plaintext_embeddings,
            'encrypted_embeddings': encrypted_embeddings
        }

        # Get answer
        print("Invoking chain...")
        response = chain.invoke(
            {"input": message},
            config={"configurable": {"session_id": session}}
        )

        answer = response.get("answer", "No answer returned")
        print(f"Answer: {answer[:200]}...")

        # Create comparison HTML
        comparison_html = create_comparison_html(
            message,
            result_contents,
            plaintext_embeddings,
            encrypted_embeddings
        )

        return answer, comparison_html

    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        print(f"ERROR in chat: {error_msg}")
        error_html = create_comparison_html(message, None, None, None)
        return f"❌ Error: {str(e)}", error_html

print("✅ Functions defined")

# Gradio UI
def get_session(request: gr.Request) -> str:
    return request.session_hash

with gr.Blocks(title="🔒 Encrypted RAG Chatbot") as demo:
    gr.Markdown("# 🔒 Encrypted RAG Chatbot with CyborgDB")
    gr.Markdown("Upload documents and ask questions - see plaintext vs encrypted vectors from search results!")

    session = gr.State(value=str(uuid.uuid4()))

    with gr.Row():
        with gr.Column(scale=2, variant="panel"):
            gr.Markdown("## 💬 Chat with Your Documents")
            system_prompt = gr.Textbox(
                label="System instruction",
                lines=2,
                value="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Keep the answer concise. {context}",
                visible=False
            )

            chatbot = gr.Chatbot(height=400, label="Conversation", type="tuples")

            with gr.Row():
                msg = gr.Textbox(
                    label="Your question",
                    placeholder="Ask a question about your documents...",
                    scale=4,
                    show_label=False
                )
                submit_btn = gr.Button("Send", variant="primary", scale=1)

            clear_btn = gr.Button("Clear Chat", size="sm")

        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("## 📄 Upload Documents")
            file_input = gr.File(
                type="filepath",
                file_count="multiple",
                label="Select PDF or TXT files"
            )
            upload_btn = gr.Button(
                "🔒 Create Encrypted Vector Store",
                variant="primary",
                size="lg"
            )
            upload_output = gr.Textbox(
                show_label=False,
                lines=6,
                placeholder="Upload files and click the button above..."
            )

            upload_btn.click(
                fn=lambda files, sess: upload_documents(files if files else [], sess),
                inputs=[file_input, session],
                outputs=[upload_output]
            )

            gr.Markdown("---")
            gr.Markdown("""
            ### 📋 Quick Start
            1. Upload documents
            2. Create encrypted vector store
            3. Ask questions
            4. See vector comparison below
            """)

    gr.Markdown("---")
    gr.Markdown("## 🔍 Retrieved Results: Vector Comparison")
    gr.Markdown("*Compare plaintext vs encrypted vectors from search results*")

    comparison_display = gr.HTML(
        create_comparison_html("Your query will appear here", None, None, None),
        label="Vector Comparison"
    )

    # Wire up chat
    def respond(message, history, sess, sys_prompt):
        answer, updated_comparison = chat(message, history, sess, sys_prompt)
        history = history + [[message, answer]]
        return "", history, updated_comparison

    msg.submit(
        respond,
        inputs=[msg, chatbot, session, system_prompt],
        outputs=[msg, chatbot, comparison_display]
    )

    submit_btn.click(
        respond,
        inputs=[msg, chatbot, session, system_prompt],
        outputs=[msg, chatbot, comparison_display]
    )

    clear_btn.click(lambda: [], None, chatbot)
    demo.load(get_session, None, session)

print("✅ Gradio interface created")

In [ ]:
# 6. Launch Gradio App

print("\n🚀 Launching Gradio app...")
demo.launch(share=True, debug=True, inline=False)
print("\n🎉 Chatbot is running! Click the public URL above.")